# 04 — Deterministic baselines and the evaluation harness

        **Estimated time:** 45 minutes<br>
        **Prerequisites:** 03 — Portable records and leakage-safe splits<br>
        **Learner-produced evidence:** validation metrics for majority and train-derived keyword baselines

        ## Learning objectives

        - Contrast a sanity-floor majority baseline with a meaningful rule baseline.
- Interpret macro and weighted classification metrics.
- Score structured output, response policy, performance, slices, and errors separately.

        This notebook is a teaching interface over the reusable code in `src/`.
        It uses only prepared local files. Run `make prepare-flight` before the
        trip; no cell installs packages or downloads data.


In [ ]:
from aai_local_finetuning.offline import enable_offline_environment

enable_offline_environment()

## Fit only on training evidence

Both baselines learn from train. Validation estimates how well a method
generalizes while prompts and settings may still change. The frozen test
remains unopened.


In [ ]:
import pandas as pd

from aai_local_finetuning.evaluation import (
    KeywordRuleBaseline,
    MajorityBaseline,
    evaluate_predictions,
    format_error_analysis,
)
from aai_local_finetuning.learning import (
    load_support_splits,
    report_row,
    support_contract,
)

splits = load_support_splits(include_test=False)
allowed_intents, _ = support_contract(splits.train)
majority = MajorityBaseline.fit(splits.train)
keyword = KeywordRuleBaseline.fit(splits.train)

## Score through one strict harness

The same evaluator parses JSON, validates the schema, rejects unsupported
labels, checks response policy, calculates classification metrics, and
summarizes latency, output tokens, memory, slices, and bounded errors.


In [ ]:
baseline_reports = {
    "majority": evaluate_predictions(
        splits.validation,
        majority.predict_many(splits.validation),
        supported_intents=allowed_intents,
    ),
    "keyword-rule": evaluate_predictions(
        splits.validation,
        keyword.predict_many(splits.validation),
        supported_intents=allowed_intents,
    ),
}
pd.DataFrame([report_row(name, report) for name, report in baseline_reports.items()])

## Why macro F1 matters

Accuracy can be dominated by frequent labels. Macro F1 gives each intent
equal weight; weighted F1 reflects observed support. Report both. The
majority method is a sanity floor and is not considered meaningful for
promotion, even when its JSON happens to be valid.


In [ ]:
pd.DataFrame(
    {
        "intent": list(baseline_reports["keyword-rule"].classification.per_intent_f1),
        "f1": list(
            baseline_reports["keyword-rule"].classification.per_intent_f1.values()
        ),
    }
).sort_values("f1").head(10)

## Inspect what the transparent baseline learned

Terms come only from training records. They are useful for debugging and
also reveal brittleness: lexical shortcuts can fail on paraphrases,
ambiguity, negation, or intents with overlapping vocabulary.


In [ ]:
{intent: keyword.keywords_by_intent[intent][:6] for intent in list(allowed_intents)[:8]}

## Bounded error evidence

Error kinds remain separate: invalid JSON, schema mismatch, unsupported
intent, intent/category/escalation errors, and response-policy failures.
Only bounded masked previews are retained.


In [ ]:
print(format_error_analysis(baseline_reports["keyword-rule"]))

## Exercise — explain the meaningful baseline

Choose one learned keyword group and predict a likely failure mode.
Success means your explanation names the shortcut and a validation slice
or example type that could expose it.


In [ ]:
inspected_intent = allowed_intents[0]
likely_failure = (
    "A paraphrase with none of the learned high-weight terms may fall back "
    "to a different intent; inspect standard and hard validation examples."
)
{
    "intent": inspected_intent,
    "keywords": keyword.keywords_by_intent[inspected_intent][:6],
    "hypothesis": likely_failure,
}

**Hint:** transparent rules are valuable because a failure hypothesis can
be tied to visible features instead of model mythology.


## Checkpoint

Record which baseline is meaningful and why valid JSON alone is not a
strong result.

**Next:** `05_prompt_baselines.ipynb` holds weights fixed and changes only
the prompt evidence on validation data.
